# Backtest Verdict - LLM-Powered Strategy Analysis

This notebook runs a hybrid verdict analysis on your trading strategy backtest results.

It combines:
- **Rule-based analysis**: Statistical rigor (Sharpe, walk-forward, robustness, beta)
- **LLM judgment**: Qwen provides nuanced assessment with suggestions

---

In [1]:
# Setup: Import the verdict function
import sys
sys.path.insert(0, '/Users/zelin/Desktop/PA Investment/Invest_strategy')

from dashboard.backend.llm_verdict import verdict
import numpy as np
import pandas as pd

## Step 1: Prepare Your Returns Data

Replace the sample data below with your actual backtest returns.

In [12]:
# OPTION A: Use your own returns (list, numpy array, or pandas Series)
# Replace this with your actual backtest returns
np.random.seed(255)
returns = np.random.randn(252) * 0.01 + 0.0005  # 1 year of daily returns

# OPTION B: Load from a file (uncomment if needed)
# returns = pd.read_csv('your_returns.csv')['returns'].values

# Benchmark returns (optional but recommended)
spy_returns = np.random.randn(252) * 0.008 + 0.0003
qqq_returns = np.random.randn(252) * 0.01 + 0.0004

print(f"Returns shape: {returns.shape}")
print(f"Mean return: {returns.mean():.4%}")
print(f"Std return: {returns.std():.4%}")

Returns shape: (252,)
Mean return: 0.0030%
Std return: 0.9949%


## Step 2: Define Your Hypothesis

A strong hypothesis is required for a meaningful verdict. Answer these questions:

In [13]:
# Fill in your strategy hypothesis
HYPOTHESIS = {
    "statement": "Momentum continues after earnings beats - stocks that beat earnings by the most continue to outperform over the next 30 days",
    "who_loses_money": "Slow traders who react late to earnings news, market makers who provide liquidity to early movers",
    "economic_mechanism": "Information diffusion - early movers have informational advantage that takes time to fully incorporate into price",
    "noise_discrimination": "Filter by beat magnitude (>5% surprise) and volume confirmation (>1.5x average volume)"
}

# Strategy parameters
N_ITERATIONS = 50       # How many parameter combinations did you test?
AVG_TURNOVER = 0.8       # Average portfolio turnover per day (0.0 to 1.0)

## Step 3: Run the Verdict

This will:
1. Run rule-based analysis (significance, walk-forward, robustness, beta)
2. Send to Qwen LLM for judgment
3. Apply override policy (LLM can tighten but not loosen verdict)

In [14]:
# Run the verdict analysis
# Set use_llm=False to skip LLM if you don't have API key configured
result = verdict(
    returns=returns,
    benchmarks={
        'SPY': spy_returns,
        'QQQ': qqq_returns
    },
    hypothesis=HYPOTHESIS["statement"],
    who_loses_money=HYPOTHESIS["who_loses_money"],
    economic_mechanism=HYPOTHESIS["economic_mechanism"],
    noise_discrimination=HYPOTHESIS["noise_discrimination"],
    n_iterations=N_ITERATIONS,
    avg_turnover=AVG_TURNOVER,
    use_llm=True,        # Set to False to skip LLM
    allow_loosening=False  # Keep False for safety
)


  VERDICT: ABANDON

📊 RULE-BASED ANALYSIS
----------------------------------------
  Sharpe:           0.05
  Probabilistic:    22.0%
  Deflated:         0.00
  Significance:     FAIL

  Walk-forward:     0.0% win rate (0 windows)
  Walk-forward:    INCONSISTENT

  Costs +100%:      Sharpe -2.50
  Slippage 25bps:   Sharpe -3.93
  Robustness:       FRAGILE

  SPY Correlation: -12.7%
  Beta Type:       ALPHA

🧠 LLM VERDICT
----------------------------------------
  Verdict:    ABANDON
  Confidence: 85%

  Reasoning:
    The strategy shows no statistical significance, with a Sharpe ratio of 0.047 and a probabilistic Sharpe of only 22%. The walk-forward analysis is inconsistent due to a 0% win rate, and the stress test...

  Flags:
    • Zero win rate in walk-forward analysis
    • Sharpe ratio well below meaningful thresholds
    • Fragile under cost and slippage scenarios

  Suggestions:
    → Re-evaluate the hypothesis with more rigorous data filtering or alternative metrics
    → Cond

## Results

In [15]:
# Print the verdict
print("=" * 60)
print(f"FINAL VERDICT: {result['verdict']}")
print("=" * 60)
print()

# Access detailed results
details = result['details']

print("📊 RULE-BASED ANALYSIS")
print("-" * 40)
rb = details['rule_based']

sig = rb['significance']
print(f"  Sharpe Ratio:       {sig['sharpe_ratio']:.2f}")
print(f"  Probabilistic:     {sig['probabilistic_sharpe']:.1%}")
print(f"  Deflated Sharpe:  {sig['deflated_sharpe']:.2f}")
print(f"  Verdict:          {sig['verdict']}")
print()

wf = rb['walkforward']
print(f"  Windows:           {wf['n_windows']}")
print(f"  Win Rate:         {wf['win_rate']:.1%}")
print(f"  Crisis Included:  {wf['crisis_included']}")
print(f"  Verdict:          {wf['verdict']}")
print()

rob = rb['robustness']
print(f"  Base Sharpe:      {rob['base_sharpe']:.2f}")
print(f"  Costs +100%:     {rob['costs_100_sharpe']:.2f}")
print(f"  Slippage 25bps:   {rob['slippage_25_sharpe']:.2f}")
print(f"  Verdict:          {rob['verdict']}")
print()

beta = rb['beta']
print(f"  SPY Correlation:  {beta['spy_correlation']:.1%}")
print(f"  Verdict:          {beta['verdict']}")

FINAL VERDICT: ABANDON

📊 RULE-BASED ANALYSIS
----------------------------------------
  Sharpe Ratio:       0.05
  Probabilistic:     22.0%
  Deflated Sharpe:  0.00
  Verdict:          FAIL

  Windows:           0
  Win Rate:         0.0%
  Crisis Included:  False
  Verdict:          INCONSISTENT

  Base Sharpe:      0.05
  Costs +100%:     -2.50
  Slippage 25bps:   -3.93
  Verdict:          FRAGILE

  SPY Correlation:  -12.7%
  Verdict:          ALPHA


In [16]:
# LLM Verdict (if enabled)
llm = details.get('llm')
if llm and not llm.get('error'):
    print("\n🧠 LLM VERDICT")
    print("-" * 40)
    print(f"  Verdict:     {llm['final_verdict']}")
    print(f"  Confidence:  {llm['confidence']:.0%}")
    print()
    print(f"  Reasoning:")
    print(f"    {llm['reasoning']}")
    print()
    
    if llm.get('flags'):
        print("  Flags:")
        for flag in llm['flags']:
            print(f"    • {flag}")
        print()
    
    if llm.get('suggestions'):
        print("  Suggestions:")
        for suggestion in llm['suggestions']:
            print(f"    → {suggestion}")
else:
    print("\n🧠 LLM verdict not available")
    if llm and llm.get('error'):
        print(f"  Error: {llm['error']}")


🧠 LLM VERDICT
----------------------------------------
  Verdict:     ABANDON
  Confidence:  85%

  Reasoning:
    The strategy shows no statistical significance, with a Sharpe ratio of 0.047 and a probabilistic Sharpe of only 22%. The walk-forward analysis is inconsistent due to a 0% win rate, and the stress tests indicate fragility under realistic transaction costs and slippage. The hypothesis, while economically plausible, is not supported by the backtest results.

  Flags:
    • Zero win rate in walk-forward analysis
    • Sharpe ratio well below meaningful thresholds
    • Fragile under cost and slippage scenarios

  Suggestions:
    → Re-evaluate the hypothesis with more rigorous data filtering or alternative metrics
    → Conduct additional out-of-sample testing with more windows to improve robustness


## Understanding the Verdict

### Verdict Types:
- **PROCEED**: Strong all-around, passes all tests clearly
- **PROCEED_WITH_CAUTION**: Good but has minor concerns (e.g., borderline metrics, beta heavy)
- **NEEDS_WORK**: Significant issues that need addressing before deployment
- **ABANDON**: Fundamental problems, likely no economic edge

### Key Metrics:
| Metric | Good | Bad |
|--------|------|-----|
| Sharpe | >1.0 | <0.5 |
| Probabilistic Sharpe | >70% | <50% |
| Walk-forward Win Rate | >60% | <50% |
| Costs +100% Sharpe | >0 | <0 |
| SPY Correlation | <30% | >70% |